# Num. Epoch = 5

## Px - scaling 0

In [2]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 34663.34it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 1.839499
mean ε  : 0.567281
median ε: 0.567174
std ε   : 0.362351
min ε   : 0.000020


In [3]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 41631.12it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 5.076696
mean ε  : 2.841393
median ε: 2.966157
std ε   : 0.958268
min ε   : 0.483417


## Px - scaling 1

In [4]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 33257.89it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 26.101330
mean ε  : 16.571619
median ε: 23.930351
std ε   : 11.390266
min ε   : 0.000004


In [5]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:02<00:00, 15061.37it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 2.809271
mean ε  : 1.600141
median ε: 1.730163
std ε   : 0.537522
min ε   : 0.227674


## Px - scaling 2

In [6]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 34426.21it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 23.542883
mean ε  : 13.230166
median ε: 17.447059
std ε   : 9.295143
min ε   : 0.000135


In [7]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 23045.95it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 10.537510
mean ε  : 4.808987
median ε: 3.487228
std ε   : 3.253078
min ε   : 0.126616


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.



# Num. Epoch = 5


## Py - scaling 0

In [8]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 31858.79it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 14.686200
mean ε  : 8.176648
median ε: 7.956354
std ε   : 4.282423
min ε   : 0.740708


In [9]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 33004.50it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 16.769481
mean ε  : 10.410849
median ε: 11.627328
std ε   : 6.371116
min ε   : 0.027898


## Py - scaling 1


In [10]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 32901.98it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 8.298862
mean ε  : 4.180811
median ε: 3.853475
std ε   : 1.831926
min ε   : 0.000001


In [11]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 35546.70it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 10.328078
mean ε  : 6.911999
median ε: 7.228568
std ε   : 3.395595
min ε   : 0.000155


## Py - scaling 2

In [12]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 34686.21it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 10.938801
mean ε  : 6.942086
median ε: 7.492651
std ε   : 2.619629
min ε   : 0.720708


In [13]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 39667.55it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 13.046614
mean ε  : 9.900026
median ε: 12.822112
std ε   : 4.164003
min ε   : 0.012666


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(X|Y) - scaling 0

In [14]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 33886.01it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 9.130030
mean ε  : 4.138111
median ε: 4.235727
std ε   : 2.728079
min ε   : 0.000010


In [15]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 38421.57it/s]



Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 24.246303
mean ε  : 14.307056
median ε: 22.471190
std ε   : 9.603158
min ε   : 0.181624


## P(X|Y) - scaling 1

In [16]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 34581.11it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 17.151693
mean ε  : 4.373114
median ε: 0.694388
std ε   : 5.561717
min ε   : 0.000012


In [17]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 27656.67it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 20.607396
mean ε  : 13.006139
median ε: 13.890147
std ε   : 7.118277
min ε   : 0.074083


## P(X|Y) - scaling 2

In [18]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:02<00:00, 20241.35it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 10.998391
mean ε  : 4.937128
median ε: 4.860940
std ε   : 2.560775
min ε   : 0.000065


In [20]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 30487.13it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 25.514261
mean ε  : 14.645820
median ε: 12.909634
std ε   : 8.305922
min ε   : 0.183969


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(Y|X) - scaling 0

In [21]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:03<00:00, 14601.01it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 2.818225
mean ε  : 1.332440
median ε: 1.447695
std ε   : 0.521979
min ε   : 0.000878


In [22]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 42321.26it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 3.472923
mean ε  : 2.240655
median ε: 2.395310
std ε   : 0.516664
min ε   : 0.465111


## P(Y|X) - scaling 1

In [23]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 35513.46it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 3.470205
mean ε  : 1.748862
median ε: 1.783578
std ε   : 0.551382
min ε   : 0.002517


In [24]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 35641.17it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 4.378520
mean ε  : 2.657206
median ε: 2.700384
std ε   : 0.546104
min ε   : 0.474682


## P(Y|X) - scaling 2

In [25]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue                      # skip anything that doesn't match
    key = tuple(map(int, m.groups())) # (client_id, round)
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# Consistency check: we need a descriptor, a feature tensor and labels
# for every (client, round) key
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Build Gaussian (mean, variance) summaries for each client-round pair
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)  # (n, d)
    mu  = X.mean(axis=0)                         # (d,)
    var = X.var(axis=0)                          # (d,)  — note: variance, not cov
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """Return W2 distance between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    diff_mean_sq = np.sum((mu1 - mu2) ** 2)
    diff_std_sq  = np.sum((np.sqrt(var1) - np.sqrt(var2)) ** 2)
    return np.sqrt(diff_mean_sq + diff_std_sq)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    mu1, var1    = gaussians[k1]
    mu2, var2    = gaussians[k2]
    ref_dist     = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print("\nWasserstein-2 ε statistics over {:d} pairs:".format(len(epsilons)))
print("-" * 40)
print("max ε   : {:.6f}".format(epsilons.max()))
print("mean ε  : {:.6f}".format(epsilons.mean()))
print("median ε: {:.6f}".format(np.median(epsilons)))
print("std ε   : {:.6f}".format(epsilons.std()))
print("min ε   : {:.6f}".format(epsilons.min()))

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 34182.07it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 3.833104
mean ε  : 2.038697
median ε: 2.059539
std ε   : 0.589762
min ε   : 0.000507


In [26]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# JSD SETUP: build a fixed global binning and per‐client histograms
# ----------------------------------------------------------------------
# 1) gather all values to find global range
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50  # you can tweak

# 2) define bin edges once
bins = np.linspace(vmin, vmax, nbins + 1)

# 3) histogram each client‐round into a probability vector
histograms = {}
for k in descriptors:
    data = flatten(features[k]).ravel()
    counts, _ = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    histograms[k] = prob

# ----------------------------------------------------------------------
# Now compute ε_js = |‖d1 - d2‖₂ − JSD(p1,p2)| over all pairs
# ----------------------------------------------------------------------
pairs     = list(itertools.combinations(descriptors.keys(), 2))
eps_js    = []

for k1, k2 in tqdm(pairs, desc="JSD pairs"):
    # profile distance
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])

    # Jensen–Shannon (scipy returns √JS divergence)
    p, q = histograms[k1], histograms[k2]
    js = jensenshannon(p, q)

    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print("\nJensen–Shannon ε statistics over {:d} pairs:".format(len(eps_js)))
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 35567.12it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 4.590942
mean ε  : 2.947128
median ε: 2.963691
std ε   : 0.588353
min ε   : 0.475847
